# Generative models

A generative machine learning model learns the underlying probability distribution of a dataset, allowing it to generate new data samples.

- Learns a mapping between noise (latent variables) and data.
- The noise (prior) and output data can have different dimensions.
- Generates new samples that resemble the original training data.

# The generative modelling trilemma

The generative modelling trilemma describes the trade-off between three desirable properties of generative models. Typically, a model performs well in two properties at the expense of the third.

1. Quality, Expressivity, Capacity - The ability to generate high-quality, diverse and complex data.
2. Steps, Time, Depth - The computational cost of generation, including the number of steps and time required.
3. Fit, Coverage - The ability to accurately model the training data distribution and cover its full range of possible samples.

For example, GANs have:

- High quality: GANs can generate realistic, high-quality images.
- Fast generation: They can generate samples in a single forward pass.
- Poor coverage: They can suffer from mode collapse, where the generator produces only a limited variety of outputs, failing to capture the full data distribution.

# Autoregressive generative models

Autoregressive models learn to generate data by maximising the likelihood of the training data.

- Split the data into a sequence of tokens.
- Predict each token based on the previously generated tokens.
- Generate data one token at a time, using each prediction as context for the next.

The joint probability of a sequence is factorised using the chain rule of probability:

$$
p_\theta(x) = p_\theta(x_1, \ldots, x_N)
= \prod_{i=1}^{N} p_\theta(x_i \mid x_1, \ldots, x_{i-1})
$$

- Each token is conditioned on all preceding tokens.

Generation is slow because predictions must be made sequentially, rather than in parallel.

# Diffusion probabilistic models

Instead of modelling a sequence of words or tokens, diffusion probabilistic models learn to reverse a gradual process of adding noise to data.

They consist of two processes:

## Forward processing (noising)

Gradually adds noise to the original data over $T$ steps, eventually producing approximately pure noise.

$$
q(x_{1:T}\mid x_0) =
\prod_{t=1}^{T} q(x_t\mid x_{t-1})
$$

- $x_0$: Original data.
- $x_t$: Data at timestep \(t\), with increasing noise.
- $q$: Forward noising process (typically fixed, not learned).

## Reverse processing (denoising)

Learns to progressively remove noise, transforming random noise back into a data sample.

$$
p_\theta(x_{0:T}) =
p(x_T)\prod_{t=1}^{T}p_\theta(x_{t-1}\mid x_t)
$$

- $x_T$: Initial random noise.
- $x_{t-1}$: Less noisy data at the previous timestep.
- $p_\theta$: Learned reverse process.

Generation requires many sequential denoising steps, making the process computationally expensive and relatively slow.

# Manifolds and embeddings

A manifold is a topological space that locally resembles Euclidean space (i.e. ordinary flat space), even if its overall structure is more complex or curved.

Common types include:

- Topological manifold - A space that locally resembles Euclidean space, with a continuous structure.
- Differentiable manifold - A manifold with a smooth structure, allowing calculus and differentiation.
- Riemannian manifold - A differentiable manifold equipped with a metric that defines distances, angles and curvature.

An embedding is a function $\phi$ that maps one manifold $M$ into another manifold $N$, while preserving its structure.
$$
\phi : M \rightarrow N
$$

- Injective: Distinct points in \(M\) map to distinct points in \(N\).
- Structure-preserving: The mapping preserves the relevant topological or geometric properties of the original manifold.
- Allows a manifold to be represented within a different space, potentially with a higher dimension.

# Energy-based models

An energy-based model (EBM) assigns an energy value to an input, indicating how compatible it is with the learned data distribution.

- Low energy: Inputs that are likely to belong to the data distribution.
- High energy: Inputs that are unlikely to belong to the data distribution.

$$
E(x) \approx 0 \quad \checkmark
$$

$$
E(\tilde{x}) > 0 \quad \times
$$

Where:

- $x$: An input from the data distribution.
- $\tilde{x}$: An input that does not belong to the data distribution.
- $E$: The energy function.

The model learns to assign lower energy to realistic data and higher energy to unrealistic data.

## Softmin and softmax

Softmax and softmin transform a set of values into a probability distribution, where:

- Each output is in the range $[0,1]$.
- All outputs sum to 1.
 
Softmax:

- Assigns higher probabilities to larger input values.
- Commonly used in classification models to convert output scores (logits) into probabilities.
- Often used in the final layer of a classifier.

Softmin:

- Assigns higher probabilities to smaller input values.
- Useful in energy-based models, where lower energy represents more likely data.

## Diffusion based approaches to energy-based models

Diffusion probabilistic models (e.g. DDPMs) commonly use a U-Net architecture to learn how to progressively remove noise from data.

1. Forward process (diffusion):
   - Gradually adds noise to the original data over $T$ timesteps.
   - Eventually, the data approximates a simple prior distribution, typically a standard Gaussian.
2. Reverse process (denoising):
   - Starts with random noise sampled from the prior distribution:
     $$p(x_T) = \mathcal{N}(x_T; 0, I)$$
   - Gradually removes noise over $T$ timesteps to generate a realistic data sample.
   - The U-Net learns to predict the noise at each timestep.
3. Conditional generation:
   - Conditioning information can be injected into the model to guide generation towards desired outputs.
   - Similar to GANs, conditioning can improve the quality and relevance of generated samples.

## Flow matching

Flow matching is a generative modelling approach that learns a continuous transformation from a noise distribution into a data distribution.

Instead of gradually removing noise through many discrete diffusion steps, it learns a velocity field that describes how samples should move from noise towards real data.

1. Continuous transformation (ODE)  
Flow matching defines an ordinary differential equation (ODE):
$$
\frac{dx_t}{dt} = v_\theta(x_t,t), \quad t\in[0,1]
$$
Where:

   - $x_0 \sim \mathcal{N}(0,I)$: Initial random noise.
   - $x_1 \sim p_{\text{data}}$: Final data sample.
   - $x_t$: Sample at an intermediate timestep.
   - $v_\theta(x_t,t)$: Learned velocity field, describing the direction and speed of movement.

2. Linear interpolation  
A simple way to define the transition between noise and data is through linear interpolation:
$$
x_t = (1-t)x_0 + tx_1
$$

   - At $t=0$: $x_t=x_0$ (pure noise).
   - At $t=1$: $x_t=x_1$ (real data).
   - Between them: $x_t$ moves along a straight line between noise and data.
3. Training objective  
The model learns to predict the velocity required to move between the noise and data samples.
$$
\mathcal{L} =
\mathbb{E}_{x_0,x_1,t}
\left[
\left\|v_\theta(x_t,t)-(x_1-x_0)\right\|^2
\right]
$$

   - $x_1-x_0$: Target velocity (direction and magnitude of movement).
   - $v_\theta(x_t,t)$: Model's predicted velocity.
   - The loss minimises the difference between the predicted and target velocities.
4. Optimal transport and straight paths  
   - Optimal transport (OT) aims to find efficient ways of moving probability mass from one distribution to another.
   - Straight transition paths avoid unnecessarily complex trajectories through the intermediate space.
   - Simpler paths can make the transformation easier to learn and require fewer sampling steps.


Key advantage: Flow matching can generate samples more efficiently by learning simpler, straighter paths between noise and data, potentially reducing generation time compared with traditional diffusion models.

In summary, flow matching is:

- A generative modelling approach that learns a continuous transformation from a noise distribution into a data distribution.
- Learns a velocity field that determines how samples move from noise towards realistic data.
- Uses straight-line interpolation between noise and data to define simple transition paths.
- Based on the idea of optimal transport, aiming for efficient paths between distributions.
- Advantage: Simpler transition paths can make generation faster and require fewer sampling steps than traditional diffusion models.